#0단계: 사전작업- 정면, 윗면 일치 이미지 리스트 출력

In [ ]:
import argparse
import csv
import os
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Iterable


DEFAULT_TOP_ROOT = Path(
    r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/2작기"
)
DEFAULT_FRONT_ROOT = Path(
    r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/0. 원본/2작기"
)
DEFAULT_OUTPUT_DIR = Path.cwd() / "outputs"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
NAME_RE = re.compile(
    r"^(?P<saved_bed>bed(?P<saved_bed_num>\d{1,3}))_"
    r"(?P<date>\d{8})_"
    r"(?P<time>\d{6})_"
    r"(?P<cam>cam[012])"
    r"(?P<extra>.*?)"
    r"(?P<ext>\.[A-Za-z0-9]+)$"
)


In [ ]:

@dataclass
class ImageRecord:
    source_view: str
    root: str
    path: str
    file_name: str
    saved_bed: str
    saved_bed_num: int
    date: str
    time: str
    datetime: str
    cam: str
    triplet_key: str
    event_time_key: str
    ext: str
    file_size: int
    parent_dir: str


def normalize_date_folder(name: str) -> str | None:
    if not re.fullmatch(r"\d{6}", name):
        return None
    return "20" + name


def iter_image_files(
    roots: list[tuple[str, Path]],
    date_min: str | None,
    date_max: str | None,
    only_date: str | None,
    max_files: int | None,
) -> Iterable[tuple[str, Path, Path]]:
    yielded = 0
    for source_view, root in roots:
        if not root.exists():
            print(f"[WARN] root does not exist: {root}")
            continue

        for dirpath, dirnames, filenames in os.walk(root):
            # Folder names are usually 260306, 260307, ...
            # Pruning here avoids touching files outside the requested date range.
            kept_dirnames = []
            for dirname in dirnames:
                full_date = normalize_date_folder(dirname)
                if only_date and full_date and full_date != only_date:
                    continue
                if date_min and full_date and full_date < date_min:
                    continue
                if date_max and full_date and full_date > date_max:
                    continue
                kept_dirnames.append(dirname)
            dirnames[:] = kept_dirnames

            for file_name in filenames:
                suffix = Path(file_name).suffix.lower()
                if suffix not in IMAGE_EXTS:
                    continue
                yield source_view, root, Path(dirpath) / file_name
                yielded += 1
                if max_files is not None and yielded >= max_files:
                    return


def parse_image(source_view: str, root: Path, path: Path) -> ImageRecord | None:
    m = NAME_RE.match(path.name)
    if not m:
        return None

    date = m.group("date")
    time = m.group("time")
    dt = datetime.strptime(date + time, "%Y%m%d%H%M%S")
    saved_bed = m.group("saved_bed")
    cam = m.group("cam")

    try:
        file_size = path.stat().st_size
    except OSError:
        file_size = -1

    return ImageRecord(
        source_view=source_view,
        root=str(root),
        path=str(path),
        file_name=path.name,
        saved_bed=saved_bed,
        saved_bed_num=int(m.group("saved_bed_num")),
        date=date,
        time=time,
        datetime=dt.isoformat(sep=" "),
        cam=cam,
        triplet_key=f"{saved_bed}_{date}_{time}",
        event_time_key=f"{date}_{time}",
        ext=m.group("ext").lower(),
        file_size=file_size,
        parent_dir=path.parent.name,
    )


def scan_records(
    top_root: Path,
    front_root: Path,
    date_min: str | None,
    date_max: str | None,
    only_date: str | None,
    max_files: int | None,
) -> tuple[list[ImageRecord], list[dict]]:
    roots = [("top", top_root), ("front", front_root)]
    records: list[ImageRecord] = []
    skipped: list[dict] = []

    for source_view, root, path in iter_image_files(roots, date_min, date_max, only_date, max_files):
        rec = parse_image(source_view, root, path)
        if rec is None:
            skipped.append(
                {
                    "reason": "filename_not_parsed",
                    "source_view": source_view,
                    "root": str(root),
                    "path": str(path),
                    "file_name": path.name,
                }
            )
            continue

        if date_min and rec.date < date_min:
            continue
        if date_max and rec.date > date_max:
            continue
        records.append(rec)

    records.sort(key=lambda r: (r.date, r.time, r.saved_bed_num, r.cam, r.path))
    return records, skipped


def join_paths(paths: list[str]) -> str:
    return " | ".join(paths)


def build_triplet_manifest(records: list[ImageRecord]) -> list[dict]:
    grouped: dict[str, list[ImageRecord]] = defaultdict(list)
    for rec in records:
        grouped[rec.triplet_key].append(rec)

    rows: list[dict] = []
    for triplet_key, items in grouped.items():
        by_cam: dict[str, list[ImageRecord]] = defaultdict(list)
        for item in items:
            by_cam[item.cam].append(item)

        first = sorted(items, key=lambda r: (r.date, r.time, r.saved_bed_num))[0]
        missing = [cam for cam in ("cam0", "cam1", "cam2") if cam not in by_cam]
        duplicate_cams = [cam for cam, cam_items in by_cam.items() if len(cam_items) > 1]

        status = "complete" if not missing else "incomplete"
        if duplicate_cams:
            status = "duplicate_" + status

        row = {
            "triplet_key": triplet_key,
            "event_time_key": first.event_time_key,
            "date": first.date,
            "time": first.time,
            "datetime": first.datetime,
            "saved_bed": first.saved_bed,
            "saved_bed_num": first.saved_bed_num,
            "has_cam0": "cam0" in by_cam,
            "has_cam1": "cam1" in by_cam,
            "has_cam2": "cam2" in by_cam,
            "cam0_count": len(by_cam.get("cam0", [])),
            "cam1_count": len(by_cam.get("cam1", [])),
            "cam2_count": len(by_cam.get("cam2", [])),
            "missing_cams": ",".join(missing),
            "duplicate_cams": ",".join(sorted(duplicate_cams)),
            "triplet_status": status,
            "cam0_path": join_paths([r.path for r in by_cam.get("cam0", [])]),
            "cam1_path": join_paths([r.path for r in by_cam.get("cam1", [])]),
            "cam2_path": join_paths([r.path for r in by_cam.get("cam2", [])]),
            # Placeholders for later OCR / correction steps.
            "cam1_tag_crop_path": "",
            "ocr_visible_bed_num": "",
            "ocr_confidence": "",
            "manual_visible_bed_num": "",
            "visible_bed_num_final": "",
            "saved_vs_visible_match": "",
            "segment_id": "",
            "real_bed_id": "",
            "qc_status": "",
            "note": "",
        }
        rows.append(row)

    rows.sort(key=lambda r: (r["date"], r["time"], r["saved_bed_num"]))
    return rows


def first_per_date_saved_bed(complete_rows: list[dict]) -> list[dict]:
    selected: dict[tuple[str, str], dict] = {}
    for row in complete_rows:
        key = (row["date"], row["saved_bed"])
        if key not in selected:
            selected[key] = row.copy()
            selected[key]["first_select_reason"] = "earliest_complete_triplet_for_date_saved_bed"
            continue
        prev = selected[key]
        if (row["date"], row["time"]) < (prev["date"], prev["time"]):
            selected[key] = row.copy()
            selected[key]["first_select_reason"] = "earliest_complete_triplet_for_date_saved_bed"

    return sorted(selected.values(), key=lambda r: (r["date"], r["saved_bed_num"], r["time"]))


def build_summary(records: list[ImageRecord], manifest_rows: list[dict], skipped: list[dict]) -> list[dict]:
    cam_counts = Counter(r.cam for r in records)
    source_counts = Counter(r.source_view for r in records)
    status_counts = Counter(r["triplet_status"] for r in manifest_rows)
    date_counts = Counter(r.date for r in records)

    rows = [
        {"metric": "parsed_image_files", "value": len(records), "note": ""},
        {"metric": "skipped_unparsed_files", "value": len(skipped), "note": ""},
        {"metric": "triplet_rows_all", "value": len(manifest_rows), "note": ""},
        {
            "metric": "triplet_rows_complete",
            "value": sum(1 for r in manifest_rows if r["triplet_status"] == "complete"),
            "note": "cam0/cam1/cam2 all present, no duplicates",
        },
    ]

    for cam in ("cam0", "cam1", "cam2"):
        rows.append({"metric": f"image_count_{cam}", "value": cam_counts.get(cam, 0), "note": ""})
    for source, count in sorted(source_counts.items()):
        rows.append({"metric": f"image_count_source_{source}", "value": count, "note": ""})
    for status, count in sorted(status_counts.items()):
        rows.append({"metric": f"triplet_status_{status}", "value": count, "note": ""})
    for date, count in sorted(date_counts.items()):
        rows.append({"metric": f"image_count_date_{date}", "value": count, "note": ""})

    return rows


def write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = list(rows[0].keys()) if rows else ["empty"]
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def write_tables(output_dir: Path, tables: dict[str, list[dict]], write_excel: bool, write_pickle: bool) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    for name, rows in tables.items():
        write_csv(output_dir / f"{name}.csv", rows)

    if write_excel or write_pickle:
        try:
            import pandas as pd
        except ImportError:
            print("[WARN] pandas is not installed. CSV files were written, but xlsx/pkl were skipped.")
            return

        if write_excel:
            for name, rows in tables.items():
                pd.DataFrame(rows).to_excel(output_dir / f"{name}.xlsx", index=False)

        if write_pickle:
            for name, rows in tables.items():
                pd.DataFrame(rows).to_pickle(output_dir / f"{name}.pkl")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Build cam0/cam1/cam2 triplet manifests.")
    parser.add_argument("--top-root", type=Path, default=DEFAULT_TOP_ROOT)
    parser.add_argument("--front-root", type=Path, default=DEFAULT_FRONT_ROOT)
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT_DIR)
    parser.add_argument("--date-min", default="20260306", help="inclusive YYYYMMDD; blank for no minimum")
    parser.add_argument("--date-max", default="20260430", help="inclusive YYYYMMDD; blank for no maximum")
    parser.add_argument("--only-date", default="", help="scan one YYYYMMDD date folder only; useful for tests")
    parser.add_argument("--max-files", type=int, default=0, help="stop after N image files; useful for tests")
    parser.add_argument("--no-excel", action="store_true", help="write only csv, not xlsx")
    parser.add_argument("--write-pickle", action="store_true", help="also write pkl cache files")
    # Use args=[] to avoid conflict with Jupyter/Colab kernel arguments
    return parser.parse_args(args=[])


def main() -> None:
    args = parse_args()
    date_min = args.date_min or None
    date_max = args.date_max or None
    only_date = args.only_date or None
    max_files = args.max_files or None

    print("[INFO] scanning image files...")
    print(f"[INFO] top_root   = {args.top_root}")
    print(f"[INFO] front_root = {args.front_root}")
    print(f"[INFO] date range = {date_min} ~ {date_max}")
    if only_date:
        print(f"[INFO] only_date  = {only_date}")
    if max_files:
        print(f"[INFO] max_files  = {max_files}")

    records, skipped = scan_records(args.top_root, args.front_root, date_min, date_max, only_date, max_files)
    inventory_rows = [asdict(r) for r in records]
    manifest_rows = build_triplet_manifest(records)
    complete_rows = [r for r in manifest_rows if r["triplet_status"] == "complete"]
    first_rows = first_per_date_saved_bed(complete_rows)
    summary_rows = build_summary(records, manifest_rows, skipped)

    tables = {
        "00_file_inventory_all": inventory_rows,
        "01_triplet_manifest_all": manifest_rows,
        "02_triplet_manifest_complete_only": complete_rows,
        "03_first_triplet_per_date_saved_bed": first_rows,
        "04_summary_counts": summary_rows,
    }
    if skipped:
        tables["99_skipped_unparsed_files"] = skipped

    write_tables(
        output_dir=args.output_dir,
        tables=tables,
        write_excel=not args.no_excel,
        write_pickle=args.write_pickle,
    )

    print("[DONE] outputs written to:")
    print(args.output_dir)
    for row in summary_rows[:8]:
        print(f"  - {row['metric']}: {row['value']}")


if __name__ == "__main__":
    main()


[INFO] scanning image files...
[INFO] top_root   = /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/2작기
[INFO] front_root = /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/0. 원본/2작기
[INFO] date range = 20260306 ~ 20260430
[DONE] outputs written to:
/content/outputs
  - parsed_image_files: 23348
  - skipped_unparsed_files: 0
  - triplet_rows_all: 8367
  - triplet_rows_complete: 7342
  - image_count_cam0: 8364
  - image_count_cam1: 7352
  - image_count_cam2: 7632
  - image_count_source_front: 7632


#1단계: 흰조각 crop 및 ocr

In [ ]:
"""
Crop right-side white bed-number pieces from all cam1 images.

This version is intended for the full 2nd-cycle OCR crop run, not only 50 test
images.

Colab-friendly points:
- stream file paths instead of keeping image objects in memory
- process images with a bounded ThreadPoolExecutor
- write manifest rows incrementally
- skip existing crops by default so interrupted runs can resume
- no montage generation

Example:
    !python _tmp_local_01_test_piece_crop_50.py --workers 4
"""

from __future__ import annotations

import argparse
import csv
import os
import re
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from itertools import islice
from pathlib import Path

from PIL import Image, ImageFile

try:
    from tqdm import tqdm
except ImportError:
    class tqdm:  # type: ignore
        def __init__(self, total=None, desc=None, unit=None):
            self.count = 0

        def __enter__(self):
            return self

        def __exit__(self, exc_type, exc, tb):
            print(f"[INFO] processed: {self.count}")

        def update(self, n=1):
            self.count += n


# Allows PIL to keep processing when a jpeg is slightly truncated.
ImageFile.LOAD_TRUNCATED_IMAGES = True

INPUT_FILE = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/2작기" )

SAVE_FILE = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/OCR_Piece/2작기/1. CROP")

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# Original cam1 images are usually 2592x1944.
# Coordinates are scaled automatically if an image has a different size.
BASE_W = 2592
BASE_H = 1944
BASE_CROP = {
    "x": 2120,
    "y": 410,
    "w": 400,
    "h": 340,
}

FILENAME_RE = re.compile(
    r"^(?P<saved_bed>bed\d{1,3})_(?P<date>\d{8})_(?P<time>\d{6})_cam1\."
    r"(jpg|jpeg|png|bmp|tif|tiff)$",
    re.IGNORECASE,
)


def normalize_bed(value: str) -> str:
    num = int(re.search(r"\d+", value).group())
    return f"bed{num:02d}"


def scaled_crop_box(img_w: int, img_h: int) -> tuple[int, int, int, int]:
    sx = img_w / BASE_W
    sy = img_h / BASE_H
    x = round(BASE_CROP["x"] * sx)
    y = round(BASE_CROP["y"] * sy)
    w = round(BASE_CROP["w"] * sx)
    h = round(BASE_CROP["h"] * sy)

    x1 = max(0, min(img_w - 1, x))
    y1 = max(0, min(img_h - 1, y))
    x2 = max(x1 + 1, min(img_w, x + w))
    y2 = max(y1 + 1, min(img_h, y + h))
    return x1, y1, x2, y2


def iter_cam1_images(input_dir: Path):
    """Yield valid cam1 image paths one by one to avoid building a large list."""
    for dirpath, _dirnames, filenames in os.walk(input_dir):
        for file_name in filenames:
            lower_name = file_name.lower()
            if "_cam1." not in lower_name:
                continue
            if Path(file_name).suffix.lower() not in IMAGE_EXTS:
                continue
            if not FILENAME_RE.match(file_name):
                continue
            yield Path(dirpath) / file_name


def count_cam1_images(input_dir: Path) -> int:
    return sum(1 for _ in iter_cam1_images(input_dir))


def crop_piece(src_path: Path, save_dir: Path, overwrite: bool) -> dict[str, str]:
    m = FILENAME_RE.match(src_path.name)
    if not m:
        return {
            "saved_bed": "",
            "date": "",
            "time": "",
            "src_path": str(src_path),
            "piece_path": "",
            "status": "skip_bad_filename",
            "error": "",
        }

    out_path = save_dir / f"{src_path.stem}_piece.jpg"
    if out_path.exists() and not overwrite:
        return {
            "saved_bed": normalize_bed(m.group("saved_bed")),
            "date": m.group("date"),
            "time": m.group("time"),
            "src_path": str(src_path),
            "piece_path": str(out_path),
            "status": "skipped_existing",
            "error": "",
        }

    try:
        with Image.open(src_path) as img:
            img = img.convert("RGB")
            crop = img.crop(scaled_crop_box(img.width, img.height))
            crop.save(out_path, quality=92, optimize=True)

        return {
            "saved_bed": normalize_bed(m.group("saved_bed")),
            "date": m.group("date"),
            "time": m.group("time"),
            "src_path": str(src_path),
            "piece_path": str(out_path),
            "status": "ok",
            "error": "",
        }
    except Exception as exc:
        return {
            "saved_bed": normalize_bed(m.group("saved_bed")),
            "date": m.group("date"),
            "time": m.group("time"),
            "src_path": str(src_path),
            "piece_path": str(out_path),
            "status": "error",
            "error": repr(exc),
        }


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Crop all cam1 bed-number pieces.")
    parser.add_argument("--input-dir", type=Path, default=INPUT_FILE)
    parser.add_argument("--save-dir", type=Path, default=SAVE_FILE)
    parser.add_argument("--workers", type=int, default=4)
    parser.add_argument("--max-in-flight", type=int, default=16)
    parser.add_argument("--overwrite", action="store_true")
    parser.add_argument("--no-count-total", action="store_true")
    parser.add_argument("--limit", type=int, default=None)
    # parse_known_args keeps the script usable inside notebooks/Colab kernels.
    args, _unknown = parser.parse_known_args()
    return args


def main() -> None:
    args = parse_args()
    input_dir = args.input_dir
    save_dir = args.save_dir
    save_dir.mkdir(parents=True, exist_ok=True)

    workers = max(1, int(args.workers))
    max_in_flight = max(workers, int(args.max_in_flight))
    manifest_path = save_dir / "piece_crop_manifest.csv"
    fieldnames = ["saved_bed", "date", "time", "src_path", "piece_path", "status", "error"]

    print(f"[INFO] INPUT_FILE     = {input_dir}")
    print(f"[INFO] SAVE_FILE      = {save_dir}")
    print(f"[INFO] workers        = {workers}")
    print(f"[INFO] max_in_flight  = {max_in_flight}")
    print(f"[INFO] overwrite      = {args.overwrite}")
    print("[INFO] scanning cam1 images recursively...")

    total_images = None
    if not args.no_count_total:
        total_images = count_cam1_images(input_dir)
        if args.limit is not None:
            total_images = min(total_images, args.limit)
        print(f"[INFO] Total cam1 images found: {total_images}")
        if total_images == 0:
            print("[INFO] No images to process. Exiting.")
            return

    counters = {"ok": 0, "skipped_existing": 0, "error": 0, "skip_bad_filename": 0}

    def submit_next(executor, iterator, futures):
        while len(futures) < max_in_flight:
            try:
                src_path = next(iterator)
            except StopIteration:
                break
            future = executor.submit(crop_piece, src_path, save_dir, args.overwrite)
            futures.add(future)

    image_iter = iter_cam1_images(input_dir)
    if args.limit is not None:
        image_iter = islice(image_iter, args.limit)

    print(f"[INFO] Processing and writing manifest to {manifest_path}")
    with manifest_path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        with ThreadPoolExecutor(max_workers=workers) as executor:
            futures = set()
            submit_next(executor, image_iter, futures)

            with tqdm(total=total_images, desc="Cropping", unit="img") as pbar:
                while futures:
                    done, futures = wait(futures, return_when=FIRST_COMPLETED)
                    for future in done:
                        row = future.result()
                        writer.writerow(row)
                        status = row.get("status", "error")
                        counters[status] = counters.get(status, 0) + 1
                        pbar.update(1)
                    f.flush()
                    submit_next(executor, image_iter, futures)

    print("[DONE] All crops finished.")
    print(f"[DONE] manifest: {manifest_path}")
    print(f"[DONE] summary : {counters}")


if __name__ == "__main__":
    main()


[INFO] INPUT_FILE     = /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/2작기
[INFO] SAVE_FILE      = /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/OCR_Piece/2작기/1. CROP
[INFO] workers        = 4
[INFO] max_in_flight  = 16
[INFO] overwrite      = False
[INFO] scanning cam1 images recursively...
[INFO] Total cam1 images found: 7352
[INFO] Processing and writing manifest to /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/OCR_Piece/2작기/1. CROP/piece_crop_manifest.csv


Cropping: 100%|██████████| 7352/7352 [21:31<00:00,  5.69img/s]

[DONE] All crops finished.
[DONE] manifest: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/OCR_Piece/2작기/1. CROP/piece_crop_manifest.csv
[DONE] summary : {'ok': 7352, 'skipped_existing': 0, 'error': 0, 'skip_bad_filename': 0}


In [ ]:
excel_files="/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/4. 결과 출력 시각화/2작기 error/outputs"